In [1]:
%pip install torch transformers accelerate

Note: you may need to restart the kernel to use updated packages.


In [38]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [39]:
text = "I am sure this project"

tokens = tokenizer.tokenize(text) # makes it into tokens 
token_ids = tokenizer.encode(text)

print("tokens:", tokens)
print("token ids:", token_ids)

tokens: ['I', 'Ġam', 'Ġsure', 'Ġthis', 'Ġproject']
token ids: [40, 1079, 2704, 419, 2390]


In [40]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

input_ids = torch.tensor([token_ids])

embedding_layer = model.model.embed_tokens
embeddings = embedding_layer(input_ids)

print("Embedding tensor shape:", embeddings.shape)
print()
print("First token embedding:")
print(embeddings[0, 0])

Loading weights: 100%|██████████| 434/434 [00:00<00:00, 3337.41it/s]


Embedding tensor shape: torch.Size([1, 5, 2048])

First token embedding:
tensor([-0.0065,  0.0243, -0.0103,  ...,  0.0204, -0.0255,  0.0103],
       dtype=torch.bfloat16, grad_fn=<SelectBackward0>)


In [41]:
layer0 = model.model.layers[0]
attention = layer0.self_attn

print("Wq:", attention.q_proj.weight.shape)
print("Wk:", attention.k_proj.weight.shape)
print("Wv:", attention.v_proj.weight.shape)

Wq: torch.Size([2048, 2048])
Wk: torch.Size([256, 2048])
Wv: torch.Size([256, 2048])


In [42]:
Q = attention.q_proj(embeddings)
K = attention.k_proj(embeddings)
V = attention.v_proj(embeddings)

print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)

Q shape: torch.Size([1, 5, 2048])
K shape: torch.Size([1, 5, 256])
V shape: torch.Size([1, 5, 256])


# we take 3 inptuts of different lengths , tokenize, then pad them together to see if it would help

In [88]:
# First get the 3 values
a = "Hello my name is qalid"
b = "small text"
c = "The FitnessGram Pacer Test is a multistage " #aerobic capacity test that progressively gets more difficult as it continues. The 20 meter pacer test will begin in 30 seconds


In [89]:
# now from text we need to essentially convert tokenes
aToken = tokenizer.tokenize(a)
bToken = tokenizer.tokenize(b)
cToken = tokenizer.tokenize(c)
# print the tokens for each
print(aToken)
print(bToken)
print(cToken)

['Hello', 'Ġmy', 'Ġname', 'Ġis', 'Ġq', 'al', 'id']
['small', 'Ġtext']
['The', 'ĠFitness', 'Gram', 'ĠP', 'acer', 'ĠTest', 'Ġis', 'Ġa', 'Ġmult', 'ist', 'age', 'Ġ']


In [90]:
# now from tokens we need to convert to ids
token_idA = tokenizer.encode(a)
token_idB = tokenizer.encode(b)
token_idC = tokenizer.encode(c)
# now print all th tokens ids -> this essentially is a intemidiate m
print(token_idA)
print(token_idB)
print(token_idC)

[9707, 847, 829, 374, 2804, 278, 307]
[9004, 1467]
[785, 35708, 64225, 393, 9584, 3393, 374, 264, 2745, 380, 424, 220]


In [91]:
# padd all of it
largestLen = max(len(token_idA),len(token_idB),len(token_idC))
# now make an array all empty and dump into them
pad_id = tokenizer.pad_token_id
if pad_id is None:
    pad_id = tokenizer.eos_token_id
newA = [pad_id] * largestLen
newB = [pad_id] * largestLen
newC = [pad_id] * largestLen

newA[:len(token_idA)] = token_idA
newB[:len(token_idB)] = token_idB
newC[:len(token_idC)] = token_idC
print(newA,len(newA))
print(newB,len(newB))
print(newC,len(newC))

[9707, 847, 829, 374, 2804, 278, 307, 151643, 151643, 151643, 151643, 151643] 12
[9004, 1467, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643] 12
[785, 35708, 64225, 393, 9584, 3393, 374, 264, 2745, 380, 424, 220] 12


In [92]:
# convert to tensors and bring it together
import torch
tensorA = torch.tensor(newA)
tensorB = torch.tensor(newB)
tensorC = torch.tensor(newC)

print(tensorA)
print(tensorB)
print(tensorC)


tensor([  9707,    847,    829,    374,   2804,    278,    307, 151643, 151643,
        151643, 151643, 151643])
tensor([  9004,   1467, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
        151643, 151643, 151643])
tensor([  785, 35708, 64225,   393,  9584,  3393,   374,   264,  2745,   380,
          424,   220])


In [93]:
# combine the tensores into one
batch = torch.stack([tensorA,tensorB,tensorC])
print(batch)

tensor([[  9707,    847,    829,    374,   2804,    278,    307, 151643, 151643,
         151643, 151643, 151643],
        [  9004,   1467, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643],
        [   785,  35708,  64225,    393,   9584,   3393,    374,    264,   2745,
            380,    424,    220]])


In [94]:
print(batch.shape)

torch.Size([3, 12])


In [95]:
# first we need to use attention masking for the padding 
attention_mask = (batch != pad_id).long()
# after padding we ask the model perdict a single token for each request 
with torch.no_grad():
    outputs = model(
        input_ids=batch,
        attention_mask=attention_mask
    )
# See the padding logits and the representation for it 
logits = outputs.logits
print(logits)
print(logits.shape)

tensor([[[16.1250,  8.3750,  3.6719,  ..., -2.1094, -2.1094, -2.1094],
         [ 6.2500,  5.5312,  4.3750,  ..., -0.4785, -0.4785, -0.4785],
         [ 9.7500, 11.0625,  7.0625,  ..., -0.9492, -0.9492, -0.9492],
         ...,
         [ 9.0000,  7.5312,  4.0938,  ..., -2.2031, -2.2031, -2.2031],
         [ 9.8750,  8.2500,  6.2500,  ..., -0.3809, -0.3809, -0.3809],
         [15.0625, 11.1875,  8.6875,  ..., -0.4355, -0.4355, -0.4355]],

        [[ 5.9375, 10.7500,  4.8750,  ..., -2.8438, -2.8438, -2.8438],
         [ 8.6875, 12.0000,  5.0312,  ..., -3.5938, -3.5938, -3.5938],
         [ 5.8438,  6.5000,  3.2344,  ..., -3.5781, -3.5781, -3.5781],
         ...,
         [ 7.2500, 11.3750,  7.3125,  ..., -3.0625, -3.0625, -3.0625],
         [ 4.5938,  8.3750,  7.4062,  ..., -3.6875, -3.6875, -3.6875],
         [ 5.0000, 10.8125,  4.5312,  ..., -4.9375, -4.9375, -4.9375]],

        [[ 3.3594,  4.8750,  4.3750,  ...,  0.2236,  0.2236,  0.2236],
         [ 6.5625,  4.0938,  3.1406,  ..., -2

In [ ]:
# shows all the next likely wor
# last real position for request A
for i in range(3):
    last_pos_a = attention_mask[i].sum() - 1

    # logits for the next token
    next_logits_a = logits[i, last_pos_a, :]

    # get top 10 token IDs
    top_values, top_ids = torch.topk(next_logits_a, k=10)

    for score, token_id in zip(top_values, top_ids):
        token_text = tokenizer.decode([token_id.item()])
        print(token_id.item(), score.item(), repr(token_text))
    print("")

323 17.5 ' and'
11 16.125 ','
600 15.3125 ' i'
13 15.3125 '.'
358 15.1875 ' I'
452 15.0625 ' al'
1154 14.5625 ' ,'
18991 14.5 ' ali'
271 14.5 '\n\n'
9544 14.4375 ' bin'

11051 23.625 ' medium'
3460 19.0 ' large'
26086 18.125 'medium'
4622 17.5 ' normal'
16767 16.875 'large'
2409 16.25 ' big'
271 16.125 '\n\n'
198 16.125 '\n'
4102 15.8125 '\xa0'
11 15.75 ','

17 27.75 '2'
16 21.0 '1'
21 19.125 '6'
18 18.0 '3'
20 17.625 '5'
19 17.25 '4'
65892 16.5 ' _____'
30743 16.5 ' ____'
23 16.375 '8'
220 15.625 ' '

